# P3 — Late Delivery Risk Scoring Demo

Load **LSTM Classifier** từ MLflow Registry, chạy inference trên Val set (chronological), xếp hạng Seller theo Risk Score.


In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import mlflow, mlflow.pytorch, mlflow.sklearn
import plotly.graph_objects as go
import plotly.figure_factory as ff
from mlflow.tracking import MlflowClient
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    classification_report, f1_score
)
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

os.environ['MLFLOW_S3_ENDPOINT_URL']  = f"http://{os.getenv('MINIO_ENDPOINT')}"
os.environ['AWS_ACCESS_KEY_ID']       = os.getenv('MINIO_ACCESS_KEY')
os.environ['AWS_SECRET_ACCESS_KEY']   = os.getenv('MINIO_SECRET_KEY')
os.environ['MLFLOW_S3_IGNORE_TLS']    = 'true'
mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))

print('✅ Setup xong.')

c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup xong.


## 1. Import hàm từ train.py (không lặp code)

In [2]:
from src.mlops.data_loader import load_late_delivery_data
from src.mlops.p3_late_delivery.train import (
    build_seller_sequences,
    find_best_threshold,
    SEQUENCE_FEATURE_COLS,
    LABEL_COL,
    TRAIN_RATIO,
    SEQ_LEN,
)
print('✅ Import xong.')

✅ Import xong.


## 2. Load dữ liệu & build sequences

In [3]:
df = load_late_delivery_data(min_weeks=SEQ_LEN + 1)
print(f'✅ {len(df)} seller-weeks | {df["seller_key"].nunique()} sellers')
print(f'   Late rate tổng thể: {df[LABEL_COL].mean()*100:.1f}%')

✅ 24221 seller-weeks | 1151 sellers
   Late rate tổng thể: 14.5%


In [4]:
X_list, y_list, label_time_keys, sellers_list = [], [], [], []

for seller_key, group in df.groupby('seller_key'):
    group = group.sort_values(['year', 'week_of_year']).reset_index(drop=True)
    if len(group) < SEQ_LEN + 1:
        continue
    for i in range(len(group) - SEQ_LEN):
        X_list.append(group[SEQUENCE_FEATURE_COLS].iloc[i: i + SEQ_LEN].values)
        y_list.append(int(group[LABEL_COL].iloc[i + SEQ_LEN]))
        label_row = group.iloc[i + SEQ_LEN]
        label_time_keys.append(int(label_row['year']) * 100 + int(label_row['week_of_year']))
        sellers_list.append(seller_key)

X              = np.array(X_list, dtype=np.float32)
y              = np.array(y_list, dtype=np.float32).reshape(-1, 1)
label_time_keys = np.array(label_time_keys)
sellers_arr    = np.array(sellers_list)

unique_keys = np.sort(np.unique(label_time_keys))
cutoff_idx  = max(1, int(len(unique_keys) * TRAIN_RATIO))
cutoff_key  = unique_keys[min(cutoff_idx, len(unique_keys) - 1)]
val_mask    = label_time_keys >= cutoff_key

X_val      = X[val_mask]
y_val      = y[val_mask]
sellers_val = sellers_arr[val_mask]
times_val   = label_time_keys[val_mask]

print(f'Val: {len(X_val)} samples | Late rate: {y_val.mean()*100:.1f}%')

Val: 5906 samples | Late rate: 21.9%


## 3. Load model + scaler từ MLflow (đúng scaler fit trên train)

In [5]:
client   = MlflowClient()
versions = client.search_model_versions("name='late_delivery_lstm'")
latest   = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
run_id   = latest.run_id
print(f'Model: version={latest.version} | run_id={run_id[:8]}...')

model  = mlflow.pytorch.load_model(f'models:/late_delivery_lstm/{latest.version}')
model.eval()

scaler = mlflow.sklearn.load_model(f'runs:/{run_id}/scaler')

print('✅ Load model + scaler thành công!')

Model: version=1 | run_id=96a2dd7c...


✅ Load model + scaler thành công!


## 4. Transform Val set & Inference

In [7]:
n_val, seq_dim, n_feat = X_val.shape

X_val_s = scaler.transform(X_val.reshape(-1, n_feat)).reshape(n_val, seq_dim, n_feat)

device = next(model.parameters()).device
with torch.no_grad():
    logits = model(torch.tensor(X_val_s).to(device))
    probs  = torch.sigmoid(logits).cpu().numpy().flatten()

truth = y_val.flatten().astype(int)

best_thr, best_f1 = find_best_threshold(truth, probs, metric='f1')
final_preds = (probs >= best_thr).astype(int)

print(f'Best threshold: {best_thr:.2f} | F1: {best_f1:.4f}')
print(classification_report(truth, final_preds, target_names=['Đúng hạn', 'Giao trễ']))

Best threshold: 0.63 | F1: 0.4462
              precision    recall  f1-score   support

    Đúng hạn       0.85      0.82      0.83      4611
    Giao trễ       0.43      0.47      0.45      1295

    accuracy                           0.75      5906
   macro avg       0.64      0.65      0.64      5906
weighted avg       0.75      0.75      0.75      5906



## 5. Bảng xếp hạng Top 20 Seller rủi ro cao nhất

In [8]:
result_df = pd.DataFrame({
    'seller_key':  sellers_val,
    'time_key':    times_val,
    'risk_score':  probs,
    'pred_late':   final_preds,
    'actual_late': truth,
})

seller_summary = (
    result_df.groupby('seller_key')
    .agg(
        avg_risk_score=('risk_score',  'mean'),
        n_weeks_val   =('risk_score',  'count'),
        n_pred_late   =('pred_late',   'sum'),
        n_actual_late =('actual_late', 'sum'),
    )
    .assign(actual_late_rate=lambda x: (x['n_actual_late'] / x['n_weeks_val'] * 100).round(1))
    .sort_values('avg_risk_score', ascending=False)
    .reset_index()
)

print('=== Top 20 Seller nguy cơ giao trễ cao nhất (Val set) ===')
seller_summary.head(20)[['seller_key','avg_risk_score','n_weeks_val','n_pred_late','n_actual_late','actual_late_rate']]

=== Top 20 Seller nguy cơ giao trễ cao nhất (Val set) ===


,seller_key,avg_risk_score,n_weeks_val,n_pred_late,n_actual_late,actual_late_rate
0,1824,0.931246,14,14,12,85.7
1,1708,0.914755,14,14,10,71.4
2,2836,0.909015,14,14,9,64.3
3,2643,0.907748,14,14,12,85.7
4,857,0.906188,14,14,11,78.6
5,1540,0.903065,10,10,8,80.0
6,1535,0.895835,14,14,9,64.3
7,368,0.894712,14,14,9,64.3
8,881,0.888991,14,14,11,78.6
9,1235,0.875938,14,14,10,71.4


## 6. ROC Curve + Confusion Matrix

In [9]:
from plotly.subplots import make_subplots

fpr, tpr, _ = roc_curve(truth, probs)
roc_auc     = auc(fpr, tpr)
cm          = confusion_matrix(truth, final_preds)
labels      = ['Đúng hạn', 'Giao trễ']

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(f'ROC Curve (AUC={roc_auc:.4f})',
                                    f'Confusion Matrix (thr={best_thr:.2f})'))

fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'LSTM AUC={roc_auc:.4f}',
                         line=dict(color='#3B82F6', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name='Random',
                         line=dict(color='gray', dash='dash')), row=1, col=1)

fig.add_trace(go.Heatmap(z=cm, x=labels, y=labels,
                         colorscale='Blues', showscale=False,
                         text=cm, texttemplate='%{text}'), row=1, col=2)

fig.update_layout(template='plotly_white', height=420,
                  title=dict(text='P3 — Đánh giá mô hình Late Delivery LSTM', x=0.5))
fig.show()

## 7. Phân bố Risk Score

In [10]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=probs[truth==0], name='Đúng hạn', opacity=0.7,
                           nbinsx=40, marker_color='#3B82F6'))
fig.add_trace(go.Histogram(x=probs[truth==1], name='Giao trễ', opacity=0.7,
                           nbinsx=40, marker_color='#EF4444'))
fig.add_vline(x=best_thr, line_dash='dash', line_color='black',
              annotation_text=f'Threshold={best_thr:.2f}')
fig.update_layout(
    title=dict(text='P3 — Phân bố Risk Score theo nhãn thực tế', x=0.5),
    xaxis_title='Risk Score', yaxis_title='Số mẫu',
    barmode='overlay', template='plotly_white', height=400
)
fig.show()